[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/13_vision_transformer_blocks.ipynb)

# 13. Vision transformer and detection blocks

이미지를 token으로 바꾸는 patchify에서 window attention과 multi-scale feature fusion까지 진행한다.

**반복 형식:** 바닐라 PyTorch 실행 → profiler로 ATen/CUDA 연산 확인 → 필요할 때만 작은 텐서로 수학적 전개를 펼친다.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)


In [ ]:
from torch.profiler import profile, ProfilerActivity

def profile_call(name, fn, *args, **kwargs):
    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    with profile(
        activities=activities,
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        out = fn(*args, **kwargs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print(f"\n[{name}] top operators")
    sort_key = "self_cuda_time_total" if torch.cuda.is_available() else "self_cpu_time_total"
    print(prof.key_averages().table(sort_by=sort_key, row_limit=12))

    return out


## 1. Patchify

unfold로 이미지를 patch token으로 바꾼다.


In [ ]:
img = torch.arange(1 * 3 * 8 * 8, dtype=torch.float32, device=device).reshape(1, 3, 8, 8)
patches = F.unfold(img, kernel_size=4, stride=4).transpose(1, 2)

print("image:", img.shape)
print("patches:", patches.shape)


In [ ]:
_ = profile_call("patchify unfold", lambda z: F.unfold(z, 4, stride=4).transpose(1, 2), img)


## 2. ViT token block

patch projection + self-attention + MLP.


In [ ]:
proj = nn.Linear(3 * 4 * 4, 16).to(device)
tokens = proj(patches)
block = nn.TransformerEncoderLayer(16, 2, 64, batch_first=True).to(device)

out = block(tokens)
print(out.shape)


In [ ]:
_ = profile_call("ViT block", block, tokens)


## 3. Window partition

feature map을 작은 window로 잘라 local attention 입력을 만든다.


In [ ]:
feat = torch.arange(1 * 4 * 4 * 8, dtype=torch.float32, device=device).reshape(1, 4, 4, 8)
ws = 2

windows = feat.view(1, 2, ws, 2, ws, 8).permute(0, 1, 3, 2, 4, 5).reshape(-1, ws * ws, 8)
print("windows:", windows.shape)


In [ ]:
def partition_windows(z):
    z = z.view(1, 2, ws, 2, ws, 8)
    z = z.permute(0, 1, 3, 2, 4, 5)
    return z.reshape(-1, ws * ws, 8)

_ = profile_call("window partition", partition_windows, feat)


## 4. Patch merging

2x2 neighboring token을 concat한 뒤 channel projection한다.


In [ ]:
x = torch.randn(1, 4, 4, 8, device=device)
merged = torch.cat(
    [x[:, 0::2, 0::2], x[:, 1::2, 0::2], x[:, 0::2, 1::2], x[:, 1::2, 1::2]],
    dim=-1,
)
reduction = nn.Linear(32, 16).to(device)
print("merged:", merged.shape, "reduced:", reduction(merged).shape)


In [ ]:
_ = profile_call("patch merge reduction", reduction, merged)


## 5. FPN-style top-down fusion

상위 resolution feature를 upsample하고 lateral feature와 더한다.


In [ ]:
low = torch.randn(1, 8, 4, 4, device=device)
high = torch.randn(1, 8, 8, 8, device=device)

up = F.interpolate(low, size=high.shape[-2:], mode="nearest")
fused = high + up

print("fused:", fused.shape)


In [ ]:
_ = profile_call("FPN fusion", lambda a, b: b + F.interpolate(a, size=b.shape[-2:], mode="nearest"), low, high)


## 6. Anchor-free center decode

작은 heatmap에서 top-k center와 offset을 읽는 detection head 핵심을 본다.


In [ ]:
heatmap = torch.tensor([[[[0.1, 0.7], [0.2, 0.9]]]], device=device)
flat = heatmap.flatten(2)
score, index = flat.topk(2, dim=-1)

H, W = heatmap.shape[-2:]
y = index // W
x = index % W

print("score:", score)
print("x:", x, "y:", y)


In [ ]:
_ = profile_call("top-k center decode", lambda z: z.flatten(2).topk(2, dim=-1), heatmap)


## References and provenance

**[13.1] ViT**
- 출처: Dosovitskiy et al., An Image is Worth 16x16 Words
- 이 노트북에서 가져온 부분: patch tokenization

**[13.2] Swin Transformer**
- 출처: Liu et al., Swin Transformer
- 이 노트북에서 가져온 부분: window attention and patch merging

**[13.3] FPN**
- 출처: Lin et al., Feature Pyramid Networks
- 이 노트북에서 가져온 부분: top-down multi-scale fusion

**[13.4] YOLO/CenterNet/RT-DETR lineages**
- 출처: practical modern detectors
- 이 노트북에서 가져온 부분: anchor-free decoding and detection heads
